# Manusia-dalam-Loop: Pintu Gerbang Pra-Tindakan, Peringkat Risiko, dan Log Audit

README untuk pelajaran ini memperkenalkan Manusia-dalam-Loop dengan petikan pendek yang meminta pengguna `LULUSKAN` atau `TOLAK` selepas ejen telah menghasilkan respons. Corak itu adalah titik permulaan yang baik, tetapi pelaksanaan HITL produksi biasanya memerlukan tiga perkara tambahan:

1. **pintu gerbang pra-tindakan** yang dijalankan **sebelum** ejen melaksanakan langkah berisiko, supaya kos, ketidakbolehbalikan, dan kelewatan tetap terkawal.
2. **peningkatan risiko**, supaya tindakan berisiko rendah dilaksanakan secara automatik, tindakan berisiko sederhana diluluskan secara berkumpulan, dan hanya tindakan berisiko tinggi yang menghalang pada manusia.
3. **log audit plus gelung semakan**, supaya setiap keputusan pintu gerbang direkodkan sebagai JSONL, dan penolakan memulakan semula ejen dengan sebab berstruktur dan bukannya hanya mencetak `Menyemak semula...`.

Nota ini membina setiap satu di atas primitif yang sama seperti `06-system-message-framework.ipynb`. Ia berjalan sepenuhnya dalam `DEMO_MODE = True` (tidak perlu input interaktif) atau dengan arahan sebenar `input()` apabila `DEMO_MODE = False`. Nota: dalam DEMO_MODE percubaan tujuan ketiga diprogramkan supaya mekanik gelung kelihatan sepenuhnya. Penyusunan semula berasaskan semakan sebenar memerlukan `DEMO_MODE = False` dan seorang operator.

**Di luar skop (ditangani dalam pelajaran lain):** pengesahan dan kawalan akses (Bahagian 06 README ancaman 2), middleware panggilan alat (Bahagian 14 MAF mendalam), corak perdebatan multi-ejen.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Corak 1: Gerbang sebelum tindakan

Petikan HITL dalam README memanggil ejen terlebih dahulu, kemudian meminta pengguna untuk meluluskan output. Itu adalah aliran **selepas tindakan**. Ejen telah pun melaksanakan, jadi kos panggilan LLM sudah dibayar, dan apa-apa kesan sampingan (emel dihantar, baris pangkalan data ditulis, komen diposkan) telah berlaku.

Aliran **sebelum tindakan** memasukkan gerbang sebelum ejen menjalankan langkah yang berisiko. Ejen mencadangkan tindakan, gerbang memutuskan sama ada untuk melaksanakan, dan hanya dengan kelulusan barulah kesan sampingan berlaku.

| Aspek | Kelulusan selepas tindakan (petikan README) | Gerbang sebelum tindakan (notebook ini) |
|---|---|---|
| Bila kelulusan dijalankan? | Selepas ejen menghasilkan output | Sebelum sebarang kesan sampingan dilaksanakan |
| Kos LLM jika ditolak | Sudah dibayar | Hanya dibayar untuk cadangan, bukan tindakan |
| Kesan sampingan yang tidak boleh dibalikkan | Mungkin (tindakan telah berlaku) | Dicegah |
| Kejelasan audit | Kelulusan adalah pernyataan cetakan | Kelulusan adalah rekod JSON dengan cap masa, tindakan, sebab |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Corak 2: Peringkat risiko

Tidak semua tindakan memerlukan kelulusan manusia. Carian hanya-baca terhadap API awam mempunyai kepentingan yang berbeza berbanding menghantar e-mel kepada pelanggan. Melayani kedua-duanya sama membazirkan perhatian operator dan memperlahankan agen.

Model 3 peringkat yang mudah:

| Peringkat | Contoh | Aliran kelulusan |
|---|---|---|
| `rendah` (hanya baca) | Cari pangkalan pengetahuan, semak pilihan penerbangan, ambil laman web awam | Laksana auto, direkod untuk audit |
| `sederhana` (mutasi murah) | Cache keputusan, tambah penombor, jadualkan peringatan | Laksana auto, tetapi semakan berkumpulan harian |
| `tinggi` (menghadap luaran atau tidak boleh diundur) | Hantar e-mel, cas kad, hantar ke saluran awam | Blok pada kelulusan manusia |

Ini adalah satu peringkat. Sistem pengeluaran sering menggunakan peringkat yang lebih halus (contohnya, tahap kebenaran AWS IAM, peringkat akses berdasarkan peranan). Versi 3 peringkat di bawah adalah versi terkecil yang berguna untuk agen yang menggabungkan tindakan hanya-baca dan tindakan berkesan sampingan.

Pengelasan di bawah menggunakan heuristik kata kunci supaya demo kekal deterministik dan murah. Dalam sistem pengeluaran anda boleh menukar ini dengan pengelasan yang dipelajari atau enjin polisi.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Corak 3: Log audit dan gelung semakan

`print("Response approved.")` bukan log audit. Untuk kepercayaan, setiap keputusan pintu gerbang harus direkodkan sebagai peristiwa berstruktur yang boleh anda pertanyakan kemudian, ulang main, atau lampirkan kepada semakan insiden.

Dua bahagian:

1. **JSONL tambah sahaja.** Satu baris setiap keputusan, dengan cap masa, tindakan, peringkat, keputusan, sebab. Mudah untuk grep, mudah untuk dihantar ke stor log sebenar kemudian.
2. **Gelung semakan pada penolakan.** Apabila pintu gerbang memulangkan `deny`, agen akan memberi arahan semula kepada dirinya dengan sebab penolakan dalam konteks, supaya cadangan seterusnya dapat mengelakkan masalah tersebut.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Sumber tambahan

Beberapa projek awam lain melaksanakan variasi corak HITL ini. Bandingkan pendekatan untuk mencari apa yang sesuai dengan tumpukan anda:

- **LangChain** pembalut alat human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): pembalut alat yang memberhentikan pelaksanaan untuk input manusia.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ menyusun semula ini): menggunakan peranan ejen khusus untuk mewakili manusia dalam perbualan multi-ejen.
- **Microsoft Agent Framework (MAF)** middleware pemanggilan fungsi ([docs](https://learn.microsoft.com/agent-framework/)): middleware yang berlari di sekitar setiap panggilan alat/fungsi, sesuai untuk logik pintu gerbang dan aliran kelulusan.

Setiap projek mengendalikan tiga sub-corak ini secara berbeza: LangChain membalutnya sebagai alat, AutoGen menggunakan peranan ejen, dan Microsoft Agent Framework menggunakan middleware pemanggilan fungsi. Baca satu atau dua pelaksanaan secara menyeluruh sebelum memilih reka bentuk untuk ejen anda sendiri.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
